In [1]:
from typing import Callable
import numpy as np
from numpy.random import MT19937
from numpy.random import RandomState, SeedSequence
from scipy.stats import moment, bootstrap, norm
import json

In [2]:
# Mersenne twister init
SEED = 6
GENERATOR = RandomState(MT19937(SeedSequence(SEED)))

# confidence level
BETA = 0.95

In [3]:
class TaskProbabilityModel:
    _theta: np.float64
    _generator: Callable

    def __init__(
        self,
        theta: np.float64,
    ) -> None:
        self._theta = theta
        self._generator = lambda size : GENERATOR.uniform(theta, 2 * theta, size)

    def get_sample(
        self,
        size: int,
    ) -> np.ndarray:
        return self._generator(size)


In [4]:
# sample generation
probability_model = TaskProbabilityModel(theta=np.float64(6))
sample = probability_model.get_sample(100)

In [5]:
# intervals' container
intervals = dict()

In [6]:
# accurate confidence interval
intervals["accurate"] = (
    sample.max() / (1 + np.pow((1 + BETA) / 2, 0.01)),
    sample.max() / (1 + np.pow((1 - BETA) / 2, 0.01))
)

In [7]:
# asymp conf interval
intervals["asymptotic"] = (
    2/3 * (sample.mean() - norm.ppf((1 + BETA) / 2) * moment(sample, order=2) / 10),
    2/3 * (sample.mean() - norm.ppf((1 - BETA) / 2) * moment(sample, order=2) / 10)
)

In [8]:
# non-parametric bootsrap confidence interval
result = bootstrap(
    sample.reshape(1, -1),
    lambda x : 2/3 * np.mean(x),
    n_resamples=10000,
    confidence_level=BETA,
    method='basic',
    rng=GENERATOR
)

intervals["bootstrap"] = (
    result.confidence_interval.low,
    result.confidence_interval.high
)

In [9]:
with open(
    r"intervals.json", "w", encoding="utf-8"
) as json_file:
    json.dump(intervals, json_file)